Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 1
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 5)
Dimensiones de Y: (52404, 1)


In [15]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012]]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 5)
Las dimensiones de testX son:  (10533, 12, 5)
Las dimensiones de valX son:  (5189, 12, 5)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

1147/1147 - 40s - 35ms/step - ia: 0.6594 - loss: 0.3594 - mae: 0.4662 - rmse: 0.5857 - smape: 0.8911 - val_ia: 0.4277 - val_loss: 0.2560 - val_mae: 0.4025 - val_rmse: 0.4669 - val_smape: 0.7994

Epoch 2/128                                           

1147/1147 - 22s - 19ms/step - ia: 0.7632 - loss: 0.2125 - mae: 0.3565 - rmse: 0.4564 - smape: 0.6891 - val_ia: 0.4799 - val_loss: 0.2142 - val_mae: 0.3719 - val_rmse: 0.4261 - val_smape: 0.7629

Epoch 3/128                                           

1147/1147 - 23s - 20ms/step - ia: 0.7990 - loss: 0.1601 - mae: 0.3080 - rmse: 0.3953 - smape: 0.6168 - val_ia: 0.5332 - val_loss: 0.1487 - val_mae: 0.3038 - val_rmse: 0.3528 - val_smape: 0.6886

Epoch 4/128                                           

1147/1147 - 19s - 17ms/step - ia: 0.8199 - loss: 0.1334 - mae: 0.2784 - rmse: 0.3608 - smape: 0.5656 - val_ia: 0.5693 - val_loss: 0.1143 - val_mae: 0.2638 - val_rmse: 0.3089 - val_smape: 0.62

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

144/144 - 69s - 478ms/step - ia: 0.6996 - loss: 0.2771 - mae: 0.3784 - rmse: 0.4743 - smape: 0.8071 - val_ia: 0.8390 - val_loss: 0.0796 - val_mae: 0.2120 - val_rmse: 0.2773 - val_smape: 0.5331

Epoch 2/16                                                                           

144/144 - 27s - 185ms/step - ia: 0.9127 - loss: 0.0386 - mae: 0.1423 - rmse: 0.1930 - smape: 0.3824 - val_ia: 0.8619 - val_loss: 0.0516 - val_mae: 0.1812 - val_rmse: 0.2275 - val_smape: 0.4582

Epoch 3/16                                                                           

144/144 - 25s - 172ms/step - ia: 0.9405 - loss: 0.0184 - mae: 0.0979 - rmse: 0.1340 - smape: 0.2785 - val_ia: 0.8888 - val_loss: 0.0336 - val_mae: 0.1490 - val_rmse: 0.1811 - val_smape: 0.4142

Epoch 4/16                                                                           

144/144 - 26s - 180ms/step - ia: 0.9501 - loss: 0.0129 - mae: 0.0825 -

...

574/574 - 8s - 14ms/step - ia: 0.9168 - loss: 0.0452 - mae: 0.1304 - rmse: 0.1738 - smape: 0.3018 - val_ia: 0.9306 - val_loss: 0.0080 - val_mae: 0.0660 - val_rmse: 0.0850 - val_smape: 0.1988

Epoch 2/8                                                                               

574/574 - 5s - 9ms/step - ia: 0.9478 - loss: 0.0139 - mae: 0.0853 - rmse: 0.1157 - smape: 0.2083 - val_ia: 0.9477 - val_loss: 0.0045 - val_mae: 0.0486 - val_rmse: 0.0644 - val_smape: 0.1345

Epoch 3/8                                                                               

574/574 - 5s - 9ms/step - ia: 0.9515 - loss: 0.0121 - mae: 0.0794 - rmse: 0.1082 - smape: 0.1917 - val_ia: 0.9492 - val_loss: 0.0042 - val_mae: 0.0478 - val_rmse: 0.0623 - val_smape: 0.1272

Epoch 4/8                                                                               

574/574 - 5s - 9ms/step - ia: 0.9534 - loss: 0.0112 - mae: 0.0763 - rmse: 0.1041 - smape: 0.1854 - val_ia: 0.9597 - val_loss: 0.0030 - val_mae: 0.0395 - val

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



4586/4586 - 47s - 10ms/step - ia: 0.9306 - loss: 0.0414 - mae: 0.0961 - rmse: 0.1178 - smape: 0.2527 - val_ia: 0.6809 - val_loss: 0.0035 - val_mae: 0.0420 - val_rmse: 0.0509 - val_smape: 0.1203

Epoch 2/16                                                                              

4586/4586 - 44s - 10ms/step - ia: 0.9650 - loss: 0.0050 - mae: 0.0505 - rmse: 0.0629 - smape: 0.1688 - val_ia: 0.6931 - val_loss: 0.0032 - val_mae: 0.0406 - val_rmse: 0.0499 - val_smape: 0.1309

Epoch 3/16                                                                              

4586/4586 - 44s - 10ms/step - ia: 0.9677 - loss: 0.0044 - mae: 0.0470 - rmse: 0.0592 - smape: 0.1578 - val_ia: 0.6559 - val_loss: 0.0038 - val_mae: 0.0471 - val_rmse: 0.0561 - val_smape: 0.1543

Epoch 4/16                                                                              

4586/4586 - 44s - 10ms/step - ia: 0.9684 - loss: 0.0042 - mae: 0.0455 - rmse: 0.0576 - smape: 0.1540 - val_ia: 0.6846 - val_loss: 0.0034 - val_ma

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



287/287 - 8s - 27ms/step - ia: 0.6098 - loss: 0.4386 - mae: 0.5098 - rmse: 0.6476 - smape: 0.9748 - val_ia: 0.7607 - val_loss: 0.1616 - val_mae: 0.3227 - val_rmse: 0.3932 - val_smape: 0.6881

Epoch 2/32                                                                              

287/287 - 3s - 11ms/step - ia: 0.7388 - loss: 0.2593 - mae: 0.3871 - rmse: 0.5078 - smape: 0.7591 - val_ia: 0.7765 - val_loss: 0.1268 - val_mae: 0.2850 - val_rmse: 0.3509 - val_smape: 0.6774

Epoch 3/32                                                                              

287/287 - 3s - 11ms/step - ia: 0.7547 - loss: 0.2379 - mae: 0.3651 - rmse: 0.4864 - smape: 0.7218 - val_ia: 0.7855 - val_loss: 0.1164 - val_mae: 0.2732 - val_rmse: 0.3341 - val_smape: 0.6472

Epoch 4/32                                                                              

287/287 - 3s - 11ms/step - ia: 0.7667 - loss: 0.2231 - mae: 0.3490 - rmse: 0.4706 - smape: 0.6853 - val_ia: 0.7697 - val_loss: 0.1301 - val_mae: 0.2844 - 

In [22]:
print(best)

{'activation': 2, 'batch': 5, 'dropout': 0.0, 'epochs': 4, 'layers': 3.0, 'learning_rate': 0.00496146976730758, 'units': 4}
